In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Trigonometric 2D', '2026')

run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_Trig2D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_Trig2D.data 2 -> PAR_MMS_Trig2D.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/MMS_Trig2D.tpl.data (byte-copy of the baseline
# dataset with $nx/$ny node-count placeholders); the baseline case above is
# the untouched L0. Each level runs in its own build/scan/<lvl>/ directory
# so that the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = 21                        # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*NX0-1, "ny": 2*NY0-1},   # cells x2 -> h/2
    "L2": {"nx": 4*NX0-3, "ny": 4*NY0-3},   # cells x4 -> h/4
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("MMS_Trig2D.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"MMS_Trig2D_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()          # trust -partition MMS_Trig2D_L1.data 2 -> PAR_*.data + Zones
    scan_cases[lvl] = c
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# The dataset as written is an invariance case, not a decaying mode: the
# uniform initial field b0 = (0, 1) satisfies u.grad(b) = 0 (u = (y, 0) is
# horizontal, By is y-uniform), its Laplacian vanishes, and the Dirichlet
# walls impose exactly (0, 1).  Both components must therefore stay at their
# initial values to machine precision -- the same "no spurious source"
# contract as the Invariance form, here with convection switched on.
t, bx = read_son(case.dataFileName_, 'SONDE_BX')
_, by = read_son(case.dataFileName_, 'SONDE_BY')

err_bx = np.max(np.abs(bx))
err_by = np.max(np.abs(by - 1.0))
print(f"max |Bx|          : {err_bx:.3e}")
print(f"max |By - 1|      : {err_by:.3e}")
print(f"final time        : {t[-1]:.6g}")
assert err_bx < 1e-10, f"Bx not preserved: {err_bx:.3e}"
assert err_by < 1e-10, f"By not preserved: {err_by:.3e}"
print("OK: uniform (0, 1) field is invariant under convection + diffusion.")


## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`21×21`),
`L1` = cells ×2 (`41×41`), `L2` = cells ×4 (`81×81`).

The dataset as written is an **invariance case** (uniform $b_0 = (0,1)$,
convection on): both components must stay at their initial values to
machine precision. The scan asserts that identity **at every level** —
like the `Invariance` form, the constant mode is a discrete identity with
no spatial error to converge, so no order is reported.

In [ ]:
def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def level_error(subdir, case_name):
    """(max |Bx|, max |By - 1|) at the scan probe."""
    _, bx = read_son_dir(subdir, case_name, "SONDE_BX")
    _, by = read_son_dir(subdir, case_name, "SONDE_BY")
    return float(np.max(np.abs(bx))), float(np.max(np.abs(by - 1.0)))

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (21, 21), "L1": (41, 41), "L2": (81, 81)}
scan_err = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      max |Bx|        max |By - 1|")
for lvl in ("L0", "L1", "L2"):
    nx, ny = scan_nodes[lvl]
    ex, ey = scan_err[lvl]
    print(f"{lvl:<6} {nx:>3}x{ny:<4} {ex:<14.3e} {ey:.3e}")
    assert ex < 1e-10, f"Bx not preserved at {lvl}: {ex:.3e}"
    assert ey < 1e-10, f"By not preserved at {lvl}: {ey:.3e}"
print("OK: uniform (0, 1) field invariance holds at every mesh level.")